# Cross-encoder for hard pairs (Business Entity Resolution)

Fine-tunes a multilingual transformer that reads both records of a candidate pair
(original name and address, including Devanagari) and scores whether they are the
same business. It is only used on the pairs the pair model is unsure about.

* **Model:** `FacebookAI/xlm-roberta-base` (MIT license, 278M parameters), trained only on the
  provided challenge data. No external data, no API calls.
* **Input** (from `python src/ce_export.py` on the Mac): upload the folder `work/ce/`
  to Google Drive as **`MyDrive/er_ce/`** with `ce_train.parquet`, `ce_oof.parquet`,
  `ce_test.parquet`.
* **Run:** Runtime > Change runtime type > **T4 GPU**, then Runtime > **Run all**
  (about 1 hour on a T4).
* **Output:** `MyDrive/er_ce/ce_scores_<name>.parquet` for every exported file.
  Download both into `student_resource/work/ce/` on the Mac.
* **If the session disconnects:** Run all again. The trained model and finished score
  chunks are saved on Drive and are not redone.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
DIR = '/content/drive/MyDrive/er_ce'
for f in ('ce_train.parquet', 'ce_oof.parquet', 'ce_test.parquet'):
    assert os.path.exists(f'{DIR}/{f}'), f'missing {DIR}/{f}: upload work/ce/ to MyDrive/er_ce first'
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'],
                     capture_output=True, text=True).stdout)

In [ ]:
import math, time
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score, log_loss
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'
MODEL_NAME = 'FacebookAI/xlm-roberta-base'   # MIT license, 278M parameters
MAX_LEN = 96                      # tokens for the pair (both records together)
BATCH, EVAL_BATCH = 32, 256
LR, EPOCHS, WARMUP = 2e-5, 1, 0.06
SEED = 42
MODEL_DIR = f'{DIR}/model'
DEV = 'cuda'
torch.manual_seed(SEED)
np.random.seed(SEED)
tok = AutoTokenizer.from_pretrained(MODEL_NAME)


def batches(df, bs, shuffle=False, seed=SEED):
    idx = np.arange(len(df))
    if shuffle:
        np.random.default_rng(seed).shuffle(idx)
    for s in range(0, len(idx), bs):
        b = df.iloc[idx[s:s + bs]]
        enc = tok(list(b.text_a), list(b.text_b), truncation='longest_first',
                  max_length=MAX_LEN, padding=True, return_tensors='pt')
        yield b, {k: v.to(DEV) for k, v in enc.items()}


@torch.no_grad()
def logits_of(model, df):
    model.eval()
    out = []
    with torch.autocast('cuda', dtype=torch.float16):
        for _, enc in batches(df, EVAL_BATCH):
            out.append(model(**enc).logits.squeeze(-1).float().cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0, dtype=np.float32)

## 1. Fine-tune (skipped if a trained model is already on Drive)
2% of the S1 entities of the fine-tuning set are held out to check the model against
the pair model's own probability on the same pairs.

In [ ]:
if os.path.exists(f'{MODEL_DIR}/config.json'):
    print('trained model found on Drive, skipping training')
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEV)
else:
    data = pd.read_parquet(f'{DIR}/ce_train.parquet')
    held = pd.util.hash_pandas_object(data.s1_id, index=False).values % 50 == 0
    val, tr = data[held], data[~held]
    print(f'train {len(tr)} pairs, validation {len(val)}, positive rate {tr.label.mean():.3f}')
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * math.ceil(len(tr) / BATCH)
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP * steps), steps)
    scaler = torch.amp.GradScaler('cuda')
    loss_fn = torch.nn.BCEWithLogitsLoss()
    step, t0 = 0, time.time()
    for ep in range(EPOCHS):
        model.train()
        for b, enc in batches(tr, BATCH, shuffle=True, seed=SEED + ep):
            y = torch.tensor(b.label.values, dtype=torch.float32, device=DEV)
            with torch.autocast('cuda', dtype=torch.float16):
                logits = model(**enc).logits.squeeze(-1)
            loss = loss_fn(logits.float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            step += 1
            if step % 500 == 0:
                print(f'step {step}/{steps}  loss {loss.item():.4f}  {time.time() - t0:.0f}s')
    model.save_pretrained(MODEL_DIR)
    tok.save_pretrained(MODEL_DIR)
    v = 1 / (1 + np.exp(-logits_of(model, val)))
    hard = val.p.between(0.02, 0.98).values
    for name, m in (('all held-out pairs', np.ones(len(val), bool)), ('hard pairs only', hard)):
        yv = val.label.values[m]
        print(f'{name}: {m.sum()} pairs | AUC cross-encoder {roc_auc_score(yv, v[m]):.4f} '
              f'vs pair model {roc_auc_score(yv, val.p.values[m]):.4f} | '
              f'log loss {log_loss(yv, np.clip(v[m], 1e-6, 1 - 1e-6)):.4f} '
              f'vs {log_loss(yv, np.clip(val.p.values[m], 1e-6, 1 - 1e-6)):.4f}')

## 2. Score the hard pairs (resumable, in chunks)
Pairs are sorted by text length so each batch needs little padding.

In [ ]:
def score(name, chunk=100_000):
    out = f'{DIR}/ce_scores_{name}.parquet'
    if os.path.exists(out):
        print('already done:', out)
        return
    df = pd.read_parquet(f'{DIR}/ce_{name}.parquet', columns=['s1_id', 'match_id', 'text_a', 'text_b'])
    order = np.argsort((df.text_a.str.len() + df.text_b.str.len()).values, kind='stable')
    df = df.iloc[order].reset_index(drop=True)
    parts, t0 = [], time.time()
    for c0 in range(0, len(df), chunk):
        part = f'{DIR}/tmp_{name}_{c0}.parquet'
        if not os.path.exists(part):
            sub = df.iloc[c0:c0 + chunk]
            pd.DataFrame({'s1_id': sub.s1_id.values, 'match_id': sub.match_id.values,
                          'ce_logit': logits_of(model, sub)}).to_parquet(part)
        parts.append(pd.read_parquet(part))
        print(f'{name}: {min(c0 + chunk, len(df))}/{len(df)}  {time.time() - t0:.0f}s')
    pd.concat(parts, ignore_index=True).to_parquet(out)
    for c0 in range(0, len(df), chunk):
        os.remove(f'{DIR}/tmp_{name}_{c0}.parquet')
    print('wrote', out)


# every exported file except the fine-tuning set: ce_oof, ce_test, and any later top-up files
names = sorted(f[3:-8] for f in os.listdir(DIR)
               if f.startswith('ce_') and f.endswith('.parquet')
               and not f.startswith('ce_scores_') and f != 'ce_train.parquet')
for name in names:
    score(name)
print('Done. Download these files from MyDrive/er_ce/ into student_resource/work/ce/ on the Mac:',
      [f'ce_scores_{n}.parquet' for n in names])